In [ ]:

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Generate 2000 Sign Language Digit Images as Flat Pixels
np.random.seed(42)
n_samples = 2000
n_pixels = 4096 # 64x64 pixels

# Create fake image pixel values (0 to 255)
pixel_data = np.random.randint(0, 100, size=(n_samples, n_pixels))
labels = np.random.randint(0, 10, size=(n_samples, 1)) # Digits 0 to 9

# Simulate patterns: add unique brightness blocks based on digit label
for i in range(n_samples):
    digit = labels[i][0]
    # Har digit ke liye specific pixel range ko high value (bright gesture) dena
    start_px = digit * 400
    end_px = start_px + 300
    pixel_data[i, start_px:end_px] = np.random.randint(180, 255)

# 2. Save into CSV Format
data_dir = r"D:\DL_PROJECTS\Project3_Sign_Language_CNN\data"
os.makedirs(data_dir, exist_ok=True)
csv_path = os.path.join(data_dir, "sign_digits.csv")

columns = [f"pixel_{i}" for i in range(n_pixels)] + ["label"]
df = pd.DataFrame(np.hstack((pixel_data, labels)), columns=columns)
df.to_csv(csv_path, index=False)

print(f"✅ Pixel Dataset Created Successfully at: {csv_path}")
print(f"Dataset Shape: {df.shape} (2000 Samples, 4096 Pixels + 1 Label)")

In [ ]:
# 1. Load CSV
csv_path = r"D:\DL_PROJECTS\Project3_Sign_Language_CNN\data\sign_digits.csv"
df = pd.read_csv(csv_path)

X = df.drop(columns=['label']).values / 255.0 # Min-Max Scaling (0 to 1 range)
y = df['label'].values

# 2. Reshape flat pixels (4096) back into Images (1 channel, 64x64 height/width)
# Shape formatting: [Batch_Size, Channels, Height, Width]
X_reshaped = X.reshape(-1, 1, 64, 64)

# 3. Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y, test_size=0.2, random_state=42)

# 4. Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long) # Multi-class ke liye Long datatype mandatory hai

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# 5. Create DataLoader
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"X_train tensor shape: {X_train_t.shape}")
print("DataLoader is ready for multi-class CNN training!")

X_train tensor shape: torch.Size([1600, 1, 64, 64])
DataLoader is ready for multi-class CNN training!


In [ ]:
class SignLanguageCNN(nn.Module):
    def __init__(self):
        super(SignLanguageCNN, self).__init__()
        
        # Conv Block 1: Input 1 channel -> 16 channels
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2) # 64x64 -> 32x32
        
        # Conv Block 2: 16 channels -> 32 channels
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2) # 32x32 -> 16x16
        
        # Fully Connected Block
        self.dropout = nn.Dropout(0.4) # 40% neurons drop out honge training ke dauran
        self.fc1 = nn.Linear(32 * 16 * 16, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10) # 10 Classes Output (Digits 0-9)
        
    def forward(self, x):
        out = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        out = self.pool2(self.relu2(self.bn2(self.conv2(out))))
        
        out = out.view(out.size(0), -1) # Flatten
        out = self.dropout(out)
        out = self.relu3(self.fc1(out))
        out = self.fc2(out) # CrossEntropyLoss handles Softmax automatically
        return out

model = SignLanguageCNN()
print(model)

SignLanguageCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc1): Linear(in_features=8192, out_features=128, bias=True)
  (relu3): ReLU()
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [ ]:
# Multi-class classification ke liye CrossEntropyLoss use hota hai
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 15
print("Starting Sign Language CNN Training...")

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}] -> Loss: {epoch_loss/len(train_loader):.4f}")

print("🎉 Model Trained Successfully!")

Starting Sign Language CNN Training...
Epoch [1/15] -> Loss: 0.1827
Epoch [2/15] -> Loss: 0.0000
Epoch [3/15] -> Loss: 0.0000
Epoch [4/15] -> Loss: 0.0000
Epoch [5/15] -> Loss: 0.0000
Epoch [6/15] -> Loss: 0.0000
Epoch [7/15] -> Loss: 0.0000
Epoch [8/15] -> Loss: 0.0000
Epoch [9/15] -> Loss: 0.0000
Epoch [10/15] -> Loss: 0.0000
Epoch [11/15] -> Loss: 0.0000
Epoch [12/15] -> Loss: 0.0000
Epoch [13/15] -> Loss: 0.0000
Epoch [14/15] -> Loss: 0.0000
Epoch [15/15] -> Loss: 0.0000
🎉 Model Trained Successfully!


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    outputs = model(X_test_t)
    # outputs mein se highest probability wale index (class) ko choose karna
    _, predicted = torch.max(outputs.data, 1)
    
    total += y_test_t.size(0)
    correct += (predicted == y_test_t).sum().item()

accuracy = (correct / total) * 100
print(f"🔥 Final Sign Language Digits Multi-class Accuracy: {accuracy:.2f}%")

🔥 Final Sign Language Digits Multi-class Accuracy: 100.00%


: 